In [1]:
import os
import pandas as pd
from IPython.display import display, HTML

# 1. Annotation file paths (relative to 04_evaluation)
DATA_DIR = "data_received/task_b" if os.path.exists("data_received/task_b/task_b_A01.csv") else "04_evaluation/data_received/task_b"
FILE_A01 = os.path.join(DATA_DIR, "task_b_A01.csv")
FILE_A02 = os.path.join(DATA_DIR, "task_b_A02.csv")

# 2. Load annotation data
df_a01 = pd.read_csv(FILE_A01)
df_a02 = pd.read_csv(FILE_A02)

print(f"Loaded A01 ({len(df_a01)} rows) and A02 ({len(df_a02)} rows)\n")

# Function to generate label distribution comparison table (A01 vs A02)
def get_comparison_table(col_name):
    # Normalize to lowercase for exact grouping
    s1 = df_a01[col_name].astype(str).str.strip().str.lower().value_counts()
    s2 = df_a02[col_name].astype(str).str.strip().str.lower().value_counts()
    p1 = (df_a01[col_name].astype(str).str.strip().str.lower().value_counts(normalize=True) * 100).round(2)
    p2 = (df_a02[col_name].astype(str).str.strip().str.lower().value_counts(normalize=True) * 100).round(2)
    
    comp_df = pd.DataFrame({
        'A01_Count': s1,
        'A01_%': p1,
        'A02_Count': s2,
        'A02_%': p2
    }).fillna(0)
    
    comp_df['A01_Count'] = comp_df['A01_Count'].astype(int)
    comp_df['A02_Count'] = comp_df['A02_Count'].astype(int)
    
    # Add total row
    total_row = pd.DataFrame({
        'A01_Count': [comp_df['A01_Count'].sum()],
        'A01_%': [100.0],
        'A02_Count': [comp_df['A02_Count'].sum()],
        'A02_%': [100.0]
    }, index=['TOTAL'])
    
    return pd.concat([comp_df, total_row])

# =========================================================================
# DISPLAY DISTRIBUTION TABLES
# =========================================================================

# 1. Distribution of SUBTYPE
display(HTML("<h3>1. Distribution of SUBTYPE (A01 vs A02)</h3>"))
display(get_comparison_table('subtype'))

# 2. Distribution of OUTPUT_TOXICITY
display(HTML("<h3>2. Distribution of OUTPUT_TOXICITY (yes / no)</h3>"))
display(get_comparison_table('output_toxicity'))

# 3. Distribution of CONTENT_PRESERVATION
display(HTML("<h3>3. Distribution of CONTENT_PRESERVATION (yes / partial / no)</h3>"))
display(get_comparison_table('content_preservation'))

# 4. Distribution of NATURALNESS
display(HTML("<h3>4. Distribution of NATURALNESS (yes / no)</h3>"))
display(get_comparison_table('naturalness'))

# 5. Distribution of REQUEST_RETENTION_HUMAN
display(HTML("<h3>5. Distribution of REQUEST_RETENTION_HUMAN</h3>"))
display(get_comparison_table('request_retention_human'))


Loaded A01 (234 rows) and A02 (234 rows)



,A01_Count,A01_%,A02_Count,A02_%
explicit,63,26.92,63,26.92
explicit_mild_profanity,42,17.95,42,17.95
contempt,41,17.52,41,17.52
belittling_competence,26,11.11,26,11.11
implicit_threat,26,11.11,26,11.11
sarcasm,22,9.40,22,9.40
passive_aggressive,14,5.98,14,5.98
TOTAL,234,100.00,234,100.00


,A01_Count,A01_%,A02_Count,A02_%
yes,140,59.83,144,61.54
no,94,40.17,90,38.46
TOTAL,234,100.00,234,100.00


,A01_Count,A01_%,A02_Count,A02_%
yes,148,63.25,130,55.56
no,44,18.80,56,23.93
partial,42,17.95,48,20.51
TOTAL,234,100.00,234,100.00


,A01_Count,A01_%,A02_Count,A02_%
yes,209,89.32,195,83.33
no,25,10.68,39,16.67
TOTAL,234,100.00,234,100.00


,A01_Count,A01_%,A02_Count,A02_%
no_clear_request,129,55.13,126,53.85
yes,60,25.64,58,24.79
no,40,17.09,48,20.51
partial,5,2.14,2,0.85
TOTAL,234,100.00,234,100.00


In [2]:
import os
import numpy as np
import pandas as pd
from sklearn.metrics import cohen_kappa_score
from IPython.display import display, HTML

# 1. Load A01 and A02 annotations
DATA_DIR = "data_received/task_b" if os.path.exists("data_received/task_b/task_b_A01.csv") else "04_evaluation/data_received/task_b"
FILE_A01 = os.path.join(DATA_DIR, "task_b_A01.csv")
FILE_A02 = os.path.join(DATA_DIR, "task_b_A02.csv")

df_a01 = pd.read_csv(FILE_A01)
df_a02 = pd.read_csv(FILE_A02)

merged_b = pd.merge(df_a01, df_a02, on="sample_id", suffixes=("_a01", "_a02"))
assert len(merged_b) == 234, "Expected exactly 234 unique pairs!"

ORDINAL_LABELS = ['no', 'partial', 'yes']

# =========================================================================
# 2. COMPUTE INTER-ANNOTATOR AGREEMENT PER CRITERION
# =========================================================================
summary_rows = []

# --- CRITERION 1: Output Toxicity (Binary) ---
t1 = merged_b['output_toxicity_a01'].str.strip().str.lower()
t2 = merged_b['output_toxicity_a02'].str.strip().str.lower()
summary_rows.append({
    'Criterion': 'output_toxicity',
    'Evaluation_Type': 'Binary (yes vs no)',
    'N_Total': 234,
    'N_Evaluated': 234,
    'N_Excluded': 0,
    'Percent_Agreement (%)': round((t1 == t2).mean() * 100, 2),
    'Unweighted_Kappa': round(cohen_kappa_score(t1, t2), 4),
    'Weighted_Kappa (Quadratic)': '-'
})

# --- CRITERION 2: Content Preservation (3-level Ordinal) ---
c1 = merged_b['content_preservation_a01'].str.strip().str.lower()
c2 = merged_b['content_preservation_a02'].str.strip().str.lower()
summary_rows.append({
    'Criterion': 'content_preservation',
    'Evaluation_Type': 'Ordinal 3 levels (no < partial < yes)',
    'N_Total': 234,
    'N_Evaluated': 234,
    'N_Excluded': 0,
    'Percent_Agreement (%)': round((c1 == c2).mean() * 100, 2),
    'Unweighted_Kappa': round(cohen_kappa_score(c1, c2, labels=ORDINAL_LABELS), 4),
    'Weighted_Kappa (Quadratic)': round(cohen_kappa_score(c1, c2, labels=ORDINAL_LABELS, weights='quadratic'), 4)
})

# --- CRITERION 3: Naturalness (Binary) ---
n1 = merged_b['naturalness_a01'].str.strip().str.lower()
n2 = merged_b['naturalness_a02'].str.strip().str.lower()
summary_rows.append({
    'Criterion': 'naturalness',
    'Evaluation_Type': 'Binary (yes vs no)',
    'N_Total': 234,
    'N_Evaluated': 234,
    'N_Excluded': 0,
    'Percent_Agreement (%)': round((n1 == n2).mean() * 100, 2),
    'Unweighted_Kappa': round(cohen_kappa_score(n1, n2), 4),
    'Weighted_Kappa (Quadratic)': '-'
})

# --- CRITERION 4A: Request Retention Binary (has_request vs no_clear_request) ---
r1 = merged_b['request_retention_human_a01'].str.strip().str.lower()
r2 = merged_b['request_retention_human_a02'].str.strip().str.lower()

r1_bin = r1.apply(lambda x: 'no_clear_request' if x == 'no_clear_request' else 'has_request')
r2_bin = r2.apply(lambda x: 'no_clear_request' if x == 'no_clear_request' else 'has_request')
summary_rows.append({
    'Criterion': 'request_has_clear_request (9A)',
    'Evaluation_Type': 'Binary (has_request vs no_clear_request)',
    'N_Total': 234,
    'N_Evaluated': 234,
    'N_Excluded': 0,
    'Percent_Agreement (%)': round((r1_bin == r2_bin).mean() * 100, 2),
    'Unweighted_Kappa': round(cohen_kappa_score(r1_bin, r2_bin), 4),
    'Weighted_Kappa (Quadratic)': '-'
})

# --- CRITERION 4B: Request Retention Ordinal (evaluated only when both agree on has_request) ---
mask_both_req = (r1 != 'no_clear_request') & (r2 != 'no_clear_request')
r1_ord = r1[mask_both_req]
r2_ord = r2[mask_both_req]
n_evaluated_req = int(mask_both_req.sum())
n_excluded_req = 234 - n_evaluated_req

summary_rows.append({
    'Criterion': 'request_retention_ordinal (9B)',
    'Evaluation_Type': 'Ordinal 3 levels (no < partial < yes)',
    'N_Total': 234,
    'N_Evaluated': n_evaluated_req,
    'N_Excluded': n_excluded_req,
    'Percent_Agreement (%)': round((r1_ord == r2_ord).mean() * 100, 2),
    'Unweighted_Kappa': round(cohen_kappa_score(r1_ord, r2_ord, labels=ORDINAL_LABELS), 4),
    'Weighted_Kappa (Quadratic)': round(cohen_kappa_score(r1_ord, r2_ord, labels=ORDINAL_LABELS, weights='quadratic'), 4)
})

# 3. Display summary metrics table
df_agreement_updated = pd.DataFrame(summary_rows)

display(HTML("<h3>=== TASK B INTER-ANNOTATOR AGREEMENT METRICS (WEIGHTED & UNWEIGHTED KAPPA) ===</h3>"))
display(df_agreement_updated)


,Criterion,Evaluation_Type,N_Total,N_Evaluated,N_Excluded,Percent_Agreement (%),Unweighted_Kappa,Weighted_Kappa (Quadratic)
0,output_toxicity,Binary (yes vs no),234,234,0,84.62,0.6777,-
1,content_preservation,Ordinal 3 levels (no < partial < yes),234,234,0,76.07,0.5778,0.8014
2,naturalness,Binary (yes vs no),234,234,0,94.02,0.7485,-
3,request_has_clear_request (9A),Binary (has_request vs no_clear_request),234,234,0,92.74,0.8535,-
4,request_retention_ordinal (9B),Ordinal 3 levels (no < partial < yes),234,98,136,94.90,0.9031,0.9731


In [3]:
import os
import pandas as pd

# 1. Load A01 and A02 annotations
DATA_DIR = "data_received/task_b" if os.path.exists("data_received/task_b/task_b_A01.csv") else "04_evaluation/data_received/task_b"
FILE_A01 = os.path.join(DATA_DIR, "task_b_A01.csv")
FILE_A02 = os.path.join(DATA_DIR, "task_b_A02.csv")

df_a01 = pd.read_csv(FILE_A01)
df_a02 = pd.read_csv(FILE_A02)

# Merge by sample_id (234 pairs)
merged = pd.merge(df_a01, df_a02, on="sample_id", suffixes=("_a01", "_a02"))

# 2. Identify disagreements across criteria
diff_tox = merged['output_toxicity_a01'] != merged['output_toxicity_a02']
diff_cont = merged['content_preservation_a01'] != merged['content_preservation_a02']
diff_nat = merged['naturalness_a01'] != merged['naturalness_a02']
diff_req = merged['request_retention_human_a01'] != merged['request_retention_human_a02']

any_diff_mask = diff_tox | diff_cont | diff_nat | diff_req
df_diff = merged[any_diff_mask].copy().reset_index(drop=True)

# 3. Create disagreement note column
def get_diff_notes(row):
    notes = []
    if row['output_toxicity_a01'] != row['output_toxicity_a02']:
        notes.append('Toxicity')
    if row['content_preservation_a01'] != row['content_preservation_a02']:
        notes.append('Content')
    if row['naturalness_a01'] != row['naturalness_a02']:
        notes.append('Naturalness')
    if row['request_retention_human_a01'] != row['request_retention_human_a02']:
        notes.append('Request')
    return ", ".join(notes)

df_diff['disagreement_on'] = df_diff.apply(get_diff_notes, axis=1)

# 4. Build adjudication sheet template
csv_table = pd.DataFrame()
csv_table['sample_id'] = df_diff['sample_id']
csv_table['disagreement_on'] = df_diff['disagreement_on']
csv_table['input_text'] = df_diff['input_text_a01']
csv_table['output_text'] = df_diff['output_text_a01']

# Group 1: Toxicity
csv_table['A01_toxicity'] = df_diff['output_toxicity_a01']
csv_table['A02_toxicity'] = df_diff['output_toxicity_a02']
csv_table['A03_toxicity'] = ""

# Group 2: Content Preservation
csv_table['A01_content'] = df_diff['content_preservation_a01']
csv_table['A02_content'] = df_diff['content_preservation_a02']
csv_table['A03_content'] = ""

# Group 3: Naturalness
csv_table['A01_naturalness'] = df_diff['naturalness_a01']
csv_table['A02_naturalness'] = df_diff['naturalness_a02']
csv_table['A03_naturalness'] = ""

# Group 4: Request Retention
csv_table['A01_request'] = df_diff['request_retention_human_a01']
csv_table['A02_request'] = df_diff['request_retention_human_a02']
csv_table['A03_request'] = ""

csv_table['adjudicator_notes'] = ""

# 5. Adjudication file path inside data_received/task_b
output_file = os.path.join(DATA_DIR, "disagreements_task_b.csv")
# Save only if file does not exist yet to prevent overwriting completed A03 labels
if not os.path.exists(output_file):
    csv_table.to_csv(output_file, index=False, encoding="utf-8-sig")

print("=== ADJUDICATION FILE STATUS ===")
print(f"- Target file: {output_file}")
print(f"- Total disagreements: {len(csv_table)} samples")
print("- Columns for adjudication: A03_toxicity, A03_content, A03_naturalness, A03_request")

display(csv_table.head(5))


=== ADJUDICATION FILE STATUS ===
- Target file: data_received/task_b\disagreements_task_b.csv
- Total disagreements: 97 samples
- Columns for adjudication: A03_toxicity, A03_content, A03_naturalness, A03_request


,sample_id,disagreement_on,input_text,output_text,A01_toxicity,A02_toxicity,A03_toxicity,A01_content,A02_content,A03_content,A01_naturalness,A02_naturalness,A03_naturalness,A01_request,A02_request,A03_request,adjudicator_notes
0,P016,Toxicity,"To Whom It May Concern, I am utterly disgusted...","To Whom It May Concern, I am utterly disgusted...",no,yes,,no,no,,no,no,,no,no,,
1,P046,Content,It’s bloody hell frustrating to see a male lea...,It's very frustrating to see a male lead handl...,no,no,,yes,partial,,yes,yes,,no_clear_request,no_clear_request,,
2,P219,Toxicity,I'm really damn frustrated that I'm still deal...,I'm really frustrated that I'm still dealing w...,yes,no,,yes,yes,,yes,yes,,yes,yes,,
3,P134,Request,"Oh please, explain to me again why I should ca...","Oh please, explain to me again why I should ca...",yes,yes,,yes,yes,,yes,yes,,no_clear_request,yes,,
4,P164,Content,"Oh joy, a software update that deletes all my ...","Oh joy, a software update that deletes all my ...",yes,yes,,partial,yes,,yes,yes,,no_clear_request,no_clear_request,,


In [4]:
import os
import pandas as pd
import csv

# 1. File paths (all relative within data folders)
DATA_DIR = "data_received/task_b" if os.path.exists("data_received/task_b/task_b_A01.csv") else "04_evaluation/data_received/task_b"
path_a01 = os.path.join(DATA_DIR, "task_b_A01.csv")
path_a02 = os.path.join(DATA_DIR, "task_b_A02.csv")
path_dis = os.path.join(DATA_DIR, "disagreements_task_b.csv")

gold_dir = "data_human_gold_label" if os.path.isdir("data_human_gold_label") else "04_evaluation/data_human_gold_label"
os.makedirs(gold_dir, exist_ok=True)
path_gold = os.path.join(gold_dir, "task_b_editbehavior_goldlabel_234.csv")

# 2. Load A01 and A02 annotations
df_a01 = pd.read_csv(path_a01, dtype=str, keep_default_na=False)
df_a02 = pd.read_csv(path_a02, dtype=str, keep_default_na=False)

# 3. Load A03 adjudication data
try:
    df_dis = pd.read_csv(path_dis, encoding='cp1252', dtype=str, keep_default_na=False)
except UnicodeDecodeError:
    df_dis = pd.read_csv(path_dis, encoding='utf-8', dtype=str, keep_default_na=False)

dis_map = df_dis.set_index('sample_id').to_dict(orient='index')

# 4. Criteria mapping for adjudication
criteria_map = [
    ('output_toxicity', 'A03_toxicity', 'gold_toxicity'),
    ('content_preservation', 'A03_content', 'gold_content'),
    ('naturalness', 'A03_naturalness', 'gold_naturalness'),
    ('request_retention_human', 'A03_request', 'gold_request'),
]

# 5. Adjudication logic:
# - If A01 == A02 -> take unanimous label
# - If A01 != A02 -> take adjudicated label from A03
# - If A03 is empty or 'unsure' -> mark as unsure
rows = []
for idx, r1 in df_a01.iterrows():
    sid = r1['sample_id']
    r2 = df_a02[df_a02['sample_id'] == sid].iloc[0]
    dis_row = dis_map.get(sid, {})
    
    row_gold = {
        'sample_id': sid,
        'input_text': r1['input_text'],
        'output_text': r1['output_text'],
        'system_id': r1['system_id'],
        'original_label': r1['original_label'],
        'band': r1['band'],
        'subtype': r1['subtype'],
    }
    
    had_disagreement = False
    has_unsure = False
    
    for col_orig, col_a03, col_gold in criteria_map:
        v1 = r1[col_orig].strip()
        v2 = r2[col_orig].strip()
        
        if v1 == v2:
            row_gold[col_gold] = v1
        else:
            had_disagreement = True
            v3 = dis_row.get(col_a03, '').strip()
            if v3 == '' or v3.lower() == 'unsure':
                row_gold[col_gold] = 'unsure'
                has_unsure = True
            else:
                row_gold[col_gold] = v3
                
    row_gold['is_adjudicated'] = had_disagreement
    row_gold['status'] = 'needs_review' if has_unsure else 'resolved'
    rows.append(row_gold)

df_gold = pd.DataFrame(rows)

# 6. Save adjudicated gold label file directly into data_human_gold_label
df_gold.to_csv(path_gold, index=False, encoding='utf-8', quoting=csv.QUOTE_MINIMAL)

print("=" * 60)
print(f"GOLD LABEL FILE CREATED: {path_gold}")
print(f"Total samples: {len(df_gold)}")
print(f"Resolution status:\n{df_gold['status'].value_counts().to_string()}")
print(f"\nSamples adjudicated by A03 (is_adjudicated):\n{df_gold['is_adjudicated'].value_counts().to_string()}")
print("=" * 60)
print("\nGold label distribution per criterion:")
for _, _, col_gold in criteria_map:
    print(f"\n--- {col_gold} ---")
    print(df_gold[col_gold].value_counts().to_string())


GOLD LABEL FILE CREATED: data_human_gold_label\task_b_editbehavior_goldlabel_234.csv
Total samples: 234
Resolution status:
status
resolved    234

Samples adjudicated by A03 (is_adjudicated):
is_adjudicated
False    137
True      97

Gold label distribution per criterion:

--- gold_toxicity ---
gold_toxicity
yes    139
no      95

--- gold_content ---
gold_content
yes        147
no          45
partial     42

--- gold_naturalness ---
gold_naturalness
yes    209
no      25

--- gold_request ---
gold_request
no_clear_request    129
yes                  60
no                   40
partial               5


In [5]:
import os
import pandas as pd
import csv

# 1. Define file paths (all relative within data folders)
gold_dir = "data_human_gold_label" if os.path.isdir("data_human_gold_label") else "04_evaluation/data_human_gold_label"
path_gold = os.path.join(gold_dir, "task_b_editbehavior_goldlabel_234.csv")

proc_dir = "data_processed/task_b" if os.path.isdir("data_processed/task_b") else "04_evaluation/data_processed/task_b"
path_map = os.path.join(proc_dir, "task_b_pair_mapping_300.csv")

path_out_300 = os.path.join(gold_dir, "task_b_system_outputs_300_goldlabel.csv")

# 2. Load 234 gold pairs and 300 mapping outputs
df_gold = pd.read_csv(path_gold, dtype=str, keep_default_na=False)
df_map = pd.read_csv(path_map, dtype=str, keep_default_na=False)

print(f"Unique pairs (Gold): {len(df_gold)}")
print(f"Mapping outputs: {len(df_map)}")

# 3. Merge gold labels into 300 outputs via pair_id = sample_id
gold_cols = ['sample_id', 'input_text', 'gold_toxicity', 'gold_content', 'gold_naturalness', 'gold_request', 'is_adjudicated']
df_300 = pd.merge(
    df_map,
    df_gold[gold_cols],
    left_on='pair_id',
    right_on='sample_id',
    how='left'
)

# 4. Reorder columns cleanly (preserving output_text_vi if present)
cols_order = [
    'system_sample_id', 'pair_id', 'input_id', 'system_id', 'seed',
    'input_text', 'output_text'
]
if 'output_text_vi' in df_300.columns:
    cols_order.append('output_text_vi')
cols_order.extend([
    'band', 'subtype', 'original_label',
    'gold_toxicity', 'gold_content', 'gold_naturalness', 'gold_request',
    'is_adjudicated'
])
cols_order = [c for c in cols_order if c in df_300.columns]
df_300 = df_300[cols_order]

# 5. Export mapped CSV directly to data_human_gold_label
df_300.to_csv(path_out_300, index=False, encoding='utf-8', quoting=csv.QUOTE_MINIMAL)

print("=" * 65)
print(f"SUCCESSFULLY MAPPED AND SAVED: {path_out_300}")
print(f"Total rows: {len(df_300)} (Missing labels: {df_300['gold_toxicity'].isnull().sum()})")
print(f"Distribution by system:\n{df_300['system_id'].value_counts().to_string()}")
print("=" * 65)

# 6. SYSTEM PERFORMANCE COMPARISON TABLE (%)
print("\n" + "=" * 65)
print("             SYSTEM PERFORMANCE COMPARISON TABLE (%)")
print("=" * 65)

criteria = [
    ('Toxicity (gold_toxicity)', 'gold_toxicity'),
    ('Content Preservation (gold_content)', 'gold_content'),
    ('Naturalness (gold_naturalness)', 'gold_naturalness'),
    ('Request Retention (gold_request)', 'gold_request')
]

for title, col in criteria:
    print(f"\n--- {title} ---")
    ct = pd.crosstab(df_300['system_id'], df_300[col], normalize='index') * 100
    print(ct.round(2).to_string())


Unique pairs (Gold): 234
Mapping outputs: 300
SUCCESSFULLY MAPPED AND SAVED: data_human_gold_label\task_b_system_outputs_300_goldlabel.csv
Total rows: 300 (Missing labels: 0)
Distribution by system:
system_id
SFT        100
ConfigC    100
FlatC      100

             SYSTEM PERFORMANCE COMPARISON TABLE (%)

--- Toxicity (gold_toxicity) ---
gold_toxicity    no   yes
system_id                
ConfigC        52.0  48.0
FlatC          46.0  54.0
SFT            31.0  69.0

--- Content Preservation (gold_content) ---
gold_content    no  partial   yes
system_id                        
ConfigC       29.0     29.0  42.0
FlatC         20.0     23.0  57.0
SFT            4.0      4.0  92.0

--- Naturalness (gold_naturalness) ---
gold_naturalness    no   yes
system_id                   
ConfigC            8.0  92.0
FlatC             10.0  90.0
SFT               15.0  85.0

--- Request Retention (gold_request) ---
gold_request    no  no_clear_request  partial   yes
system_id                         

## System Performance Evaluation on Human Gold Labels


### 1. Performance Rates by Difficulty Band


In [6]:
import os
import pandas as pd
import numpy as np

# 1. Load 300 outputs with gold labels from data_human_gold_label
gold_dir = "data_human_gold_label" if os.path.isdir("data_human_gold_label") else "04_evaluation/data_human_gold_label"
path_300 = os.path.join(gold_dir, "task_b_system_outputs_300_goldlabel.csv")
df = pd.read_csv(path_300, keep_default_na=False)

print("=" * 80)
print("STEP 1: COMPREHENSIVE SANITY CHECK & VALIDATION")
print("=" * 80)

allowed = {
    'gold_toxicity': {'yes', 'no'},
    'gold_content': {'yes', 'partial', 'no'},
    'gold_naturalness': {'yes', 'no'},
    'gold_request': {'yes', 'partial', 'no', 'no_clear_request'}
}

validation_passed = True

# 1.1. Check for invalid labels, empty values, or lingering unsure flags
for col, valid_labels in allowed.items():
    df[col] = df[col].astype(str).str.strip()
    actual_labels = set(df[col].unique())
    invalid_labels = actual_labels - valid_labels
    has_empty = "" in actual_labels
    
    if invalid_labels or has_empty:
        validation_passed = False
        print(f"[ERROR] Column '{col}' contains invalid labels or empty values: {invalid_labels}")
    else:
        print(f"[OK] Column '{col:<22}' valid: {actual_labels}")

# 1.2. Sample count distribution by system
system_counts = df['system_id'].value_counts()
print(f"\nSample count distribution by system:\n{system_counts.to_string()}")

# 1.3. Safety assertion
assert validation_passed, "STOP! Data contains empty, invalid, or unsure labels!"
print("\n[OK] SANITY CHECK COMPLETE: Data verified, ready to compute raw rates.")

print("\n" + "=" * 80)
print("STEP 2: COMPUTE RAW UNWEIGHTED HUMAN RATES PER SYSTEM")
print("=" * 80)

master_rows = []
systems = df['system_id'].unique()

for sys_name in systems:
    sub = df[df['system_id'] == sys_name]
    
    n_sys = len(sub)
    n_tox = sub['gold_toxicity'].isin(allowed['gold_toxicity']).sum()
    n_cnt = sub['gold_content'].isin(allowed['gold_content']).sum()
    n_nat = sub['gold_naturalness'].isin(allowed['gold_naturalness']).sum()
    
    tox_counts = sub['gold_toxicity'].value_counts()
    cnt_counts = sub['gold_content'].value_counts()
    nat_counts = sub['gold_naturalness'].value_counts()
    
    valid_req_labels = {'yes', 'partial', 'no'}
    sub_eligible = sub[sub['gold_request'].isin(valid_req_labels)]
    n_eligible = len(sub_eligible)
    req_counts = sub_eligible['gold_request'].value_counts()
    
    master_rows.append({
        'System': sys_name,
        'N_total': n_sys,
        'Non-toxic (%)': round(tox_counts.get('no', 0) / n_tox * 100, 2) if n_tox > 0 else 0.0,
        'Toxic (%)': round(tox_counts.get('yes', 0) / n_tox * 100, 2) if n_tox > 0 else 0.0,
        'Content Yes (%)': round(cnt_counts.get('yes', 0) / n_cnt * 100, 2) if n_cnt > 0 else 0.0,
        'Content Part (%)': round(cnt_counts.get('partial', 0) / n_cnt * 100, 2) if n_cnt > 0 else 0.0,
        'Content No (%)': round(cnt_counts.get('no', 0) / n_cnt * 100, 2) if n_cnt > 0 else 0.0,
        'Natural Yes (%)': round(nat_counts.get('yes', 0) / n_nat * 100, 2) if n_nat > 0 else 0.0,
        'Natural No (%)': round(nat_counts.get('no', 0) / n_nat * 100, 2) if n_nat > 0 else 0.0,
        'Req N eligible': n_eligible,
        'Req Yes (%)': round(req_counts.get('yes', 0) / n_eligible * 100, 2) if n_eligible > 0 else 0.0,
        'Req Part (%)': round(req_counts.get('partial', 0) / n_eligible * 100, 2) if n_eligible > 0 else 0.0,
        'Req No (%)': round(req_counts.get('no', 0) / n_eligible * 100, 2) if n_eligible > 0 else 0.0,
    })

df_raw_master = pd.DataFrame(master_rows)

print("\nRAW UNWEIGHTED HUMAN RATES (PER SYSTEM):")
print(df_raw_master.to_string(index=False))

# Detailed breakdown by band per system
print("\n" + "=" * 135)
print("DETAILED RAW HUMAN RATES BY DIFFICULTY BAND PER SYSTEM")
print("=" * 135)

bands_list = ['easy', 'medium', 'hard']

for sys_name in systems:
    sub_sys = df[df['system_id'] == sys_name]
    table_rows = []
    
    for b in bands_list:
        sub = sub_sys[sub_sys['band'] == b]
        n_b = len(sub)
        
        n_tox = sub['gold_toxicity'].isin(allowed['gold_toxicity']).sum()
        tox_counts = sub['gold_toxicity'].value_counts()
        
        n_cnt = sub['gold_content'].isin(allowed['gold_content']).sum()
        cnt_counts = sub['gold_content'].value_counts()
        
        n_nat = sub['gold_naturalness'].isin(allowed['gold_naturalness']).sum()
        nat_counts = sub['gold_naturalness'].value_counts()
        
        sub_req = sub[sub['gold_request'].isin(valid_req_labels)]
        n_eligible = len(sub_req)
        req_counts = sub_req['gold_request'].value_counts()
        
        table_rows.append({
            'Band': b,
            'N': n_b,
            'Non-toxic (%)': round(tox_counts.get('no', 0) / n_tox * 100, 2) if n_tox > 0 else 0.0,
            'Toxic (%)': round(tox_counts.get('yes', 0) / n_tox * 100, 2) if n_tox > 0 else 0.0,
            'Content Yes (%)': round(cnt_counts.get('yes', 0) / n_cnt * 100, 2) if n_cnt > 0 else 0.0,
            'Content Part (%)': round(cnt_counts.get('partial', 0) / n_cnt * 100, 2) if n_cnt > 0 else 0.0,
            'Content No (%)': round(cnt_counts.get('no', 0) / n_cnt * 100, 2) if n_cnt > 0 else 0.0,
            'Natural Yes (%)': round(nat_counts.get('yes', 0) / n_nat * 100, 2) if n_nat > 0 else 0.0,
            'Natural No (%)': round(nat_counts.get('no', 0) / n_nat * 100, 2) if n_nat > 0 else 0.0,
            'Req N eligible': n_eligible,
            'Req Yes (%)': round(req_counts.get('yes', 0) / n_eligible * 100, 2) if n_eligible > 0 else 0.0,
            'Req Part (%)': round(req_counts.get('partial', 0) / n_eligible * 100, 2) if n_eligible > 0 else 0.0,
            'Req No (%)': round(req_counts.get('no', 0) / n_eligible * 100, 2) if n_eligible > 0 else 0.0,
        })
        
    n_tot = len(sub_sys)
    n_tox = sub_sys['gold_toxicity'].isin(allowed['gold_toxicity']).sum()
    n_cnt = sub_sys['gold_content'].isin(allowed['gold_content']).sum()
    n_nat = sub_sys['gold_naturalness'].isin(allowed['gold_naturalness']).sum()
    tox_counts = sub_sys['gold_toxicity'].value_counts()
    cnt_counts = sub_sys['gold_content'].value_counts()
    nat_counts = sub_sys['gold_naturalness'].value_counts()
    sub_eligible = sub_sys[sub_sys['gold_request'].isin(valid_req_labels)]
    n_eligible = len(sub_eligible)
    req_counts = sub_eligible['gold_request'].value_counts()
    
    table_rows.append({
        'Band': 'Overall (Raw)',
        'N': n_tot,
        'Non-toxic (%)': round(tox_counts.get('no', 0) / n_tox * 100, 2) if n_tox > 0 else 0.0,
        'Toxic (%)': round(tox_counts.get('yes', 0) / n_tox * 100, 2) if n_tox > 0 else 0.0,
        'Content Yes (%)': round(cnt_counts.get('yes', 0) / n_cnt * 100, 2) if n_cnt > 0 else 0.0,
        'Content Part (%)': round(cnt_counts.get('partial', 0) / n_cnt * 100, 2) if n_cnt > 0 else 0.0,
        'Content No (%)': round(cnt_counts.get('no', 0) / n_cnt * 100, 2) if n_cnt > 0 else 0.0,
        'Natural Yes (%)': round(nat_counts.get('yes', 0) / n_nat * 100, 2) if n_nat > 0 else 0.0,
        'Natural No (%)': round(nat_counts.get('no', 0) / n_nat * 100, 2) if n_nat > 0 else 0.0,
        'Req N eligible': n_eligible,
        'Req Yes (%)': round(req_counts.get('yes', 0) / n_eligible * 100, 2) if n_eligible > 0 else 0.0,
        'Req Part (%)': round(req_counts.get('partial', 0) / n_eligible * 100, 2) if n_eligible > 0 else 0.0,
        'Req No (%)': round(req_counts.get('no', 0) / n_eligible * 100, 2) if n_eligible > 0 else 0.0,
    })
    
    df_sys_table = pd.DataFrame(table_rows)
    print(f"\n--- RAW RATES FOR SYSTEM: {sys_name} ---")
    print(df_sys_table.to_string(index=False))


STEP 1: COMPREHENSIVE SANITY CHECK & VALIDATION
[OK] Column 'gold_toxicity         ' valid: {'yes', 'no'}
[OK] Column 'gold_content          ' valid: {'yes', 'partial', 'no'}
[OK] Column 'gold_naturalness      ' valid: {'yes', 'no'}
[OK] Column 'gold_request          ' valid: {'yes', 'partial', 'no_clear_request', 'no'}

Sample count distribution by system:
system_id
SFT        100
ConfigC    100
FlatC      100

[OK] SANITY CHECK COMPLETE: Data verified, ready to compute raw rates.

STEP 2: COMPUTE RAW UNWEIGHTED HUMAN RATES PER SYSTEM

RAW UNWEIGHTED HUMAN RATES (PER SYSTEM):
 System  N_total  Non-toxic (%)  Toxic (%)  Content Yes (%)  Content Part (%)  Content No (%)  Natural Yes (%)  Natural No (%)  Req N eligible  Req Yes (%)  Req Part (%)  Req No (%)
    SFT      100           31.0       69.0             92.0               4.0             4.0             85.0            15.0              43        90.70          0.00        9.30
ConfigC      100           52.0       48.0          

### 2. Post-Stratified Population Weighted Rates


In [7]:
import os
import pandas as pd

# 1. Path to test.csv (relative path to benchmark test split)
path_test = "../02_data_preparation/data/03_stratified_benchmark/test.csv" if os.path.exists("../02_data_preparation/data/03_stratified_benchmark/test.csv") else "02_data_preparation/data/03_stratified_benchmark/test.csv"

# 2. Load dataset
df_test = pd.read_csv(path_test)

# 3. Filter toxic test instances (population)
df_toxic = df_test[df_test['label'] == 'toxic']

# 4. Compute count and distribution (%) per difficulty band
band_counts = df_toxic['band'].value_counts()
band_pcts = df_toxic['band'].value_counts(normalize=True) * 100

# 5. Build distribution summary table
df_band_dist = pd.DataFrame({
    'Count (N)': band_counts,
    'Distribution (%)': band_pcts.round(2),
    'Weight W': (band_counts / len(df_toxic)).round(4)
})

print("=" * 55)
print(f"TOTAL TOXIC TEST INSTANCES: {len(df_toxic)} / {len(df_test)}")
print("=" * 55)
print("BAND DISTRIBUTION IN POPULATION TEST SET (TOXIC):")
print(df_band_dist.to_string())
print("=" * 55)


TOTAL TOXIC TEST INSTANCES: 1150 / 1312
BAND DISTRIBUTION IN POPULATION TEST SET (TOXIC):
        Count (N)  Distribution (%)  Weight W
band                                         
easy          518             45.04    0.4504
medium        336             29.22    0.2922
hard          296             25.74    0.2574


In [8]:
import os
import pandas as pd

# 1. Load 300 outputs with gold labels from data_human_gold_label
gold_dir = "data_human_gold_label" if os.path.isdir("data_human_gold_label") else "04_evaluation/data_human_gold_label"
path_300 = os.path.join(gold_dir, "task_b_system_outputs_300_goldlabel.csv")
df_300 = pd.read_csv(path_300, keep_default_na=False)

# 2. Extract unique inputs (deduplicated across 3 systems)
df_unique_inputs = df_300.drop_duplicates(subset=['input_id'])
N_sample_input = len(df_unique_inputs)

# 3. Group by band and compute sample proportions
band_group = df_unique_inputs.groupby('band').size()
df_sample_band = pd.DataFrame({
    'N_sample_band': band_group,
    'p_sample': (band_group / N_sample_input).round(4),
    'Percentage (%)': ((band_group / N_sample_input) * 100).round(2)
})

# 4. Print results
print("=" * 60)
print(f"TOTAL UNIQUE INPUTS IN HUMAN EVALUATION (Task B): {N_sample_input}")
print("=" * 60)
print(df_sample_band.to_string())
print("=" * 60)


TOTAL UNIQUE INPUTS IN HUMAN EVALUATION (Task B): 100
        N_sample_band  p_sample  Percentage (%)
band                                           
easy               33      0.33            33.0
hard               34      0.34            34.0
medium             33      0.33            33.0


In [9]:
import os
import pandas as pd

# 1. File paths (clean relative paths)
path_test = "../02_data_preparation/data/03_stratified_benchmark/test.csv" if os.path.exists("../02_data_preparation/data/03_stratified_benchmark/test.csv") else "02_data_preparation/data/03_stratified_benchmark/test.csv"
gold_dir = "data_human_gold_label" if os.path.isdir("data_human_gold_label") else "04_evaluation/data_human_gold_label"
path_300 = os.path.join(gold_dir, "task_b_system_outputs_300_goldlabel.csv")

# 2. Population test set (toxic samples only)
df_test = pd.read_csv(path_test)
df_test_toxic = df_test[df_test['label'] == 'toxic']
n_test_total = len(df_test_toxic)
test_counts = df_test_toxic.groupby('band').size()
p_test = test_counts / n_test_total

# 3. Human sample (unique inputs)
df_300 = pd.read_csv(path_300, keep_default_na=False)
df_unique_inputs = df_300.drop_duplicates(subset=['input_id'])
n_sample_total = len(df_unique_inputs)
sample_counts = df_unique_inputs.groupby('band').size()
p_sample = sample_counts / n_sample_total

# 4. Fixed band ordering
bands_order = ['easy', 'medium', 'hard']
p_test_raw = p_test.reindex(bands_order)
p_sample_raw = p_sample.reindex(bands_order)

# 5. Build comparison table and compute weights directly: weight = p_test / p_sample
df_band_compare = pd.DataFrame({
    'band': bands_order,
    'N_test': test_counts.reindex(bands_order).values,
    'p_test': p_test_raw.round(4).values,
    'N_sample': sample_counts.reindex(bands_order).values,
    'p_sample': p_sample_raw.round(4).values,
    'weight (p_test/p_sample)': (p_test_raw.values / p_sample_raw.values).round(4)
})

print("=" * 80)
print("BAND DISTRIBUTION COMPARISON: POPULATION TEST (TOXIC) VS HUMAN SAMPLE")
print("=" * 80)
print(df_band_compare.to_string(index=False))
print("=" * 80)
print(f"Total N_test (toxic): {n_test_total} | Total N_sample (unique): {n_sample_total}")


BAND DISTRIBUTION COMPARISON: POPULATION TEST (TOXIC) VS HUMAN SAMPLE
  band  N_test  p_test  N_sample  p_sample  weight (p_test/p_sample)
  easy     518  0.4504        33      0.33                    1.3650
medium     336  0.2922        33      0.33                    0.8854
  hard     296  0.2574        34      0.34                    0.7570
Total N_test (toxic): 1150 | Total N_sample (unique): 100


In [10]:
import os
import pandas as pd
import numpy as np

# 1. File paths (clean relative paths)
path_test = "../02_data_preparation/data/03_stratified_benchmark/test.csv" if os.path.exists("../02_data_preparation/data/03_stratified_benchmark/test.csv") else "02_data_preparation/data/03_stratified_benchmark/test.csv"
gold_dir = "data_human_gold_label" if os.path.isdir("data_human_gold_label") else "04_evaluation/data_human_gold_label"
path_300 = os.path.join(gold_dir, "task_b_system_outputs_300_goldlabel.csv")

# ==============================================================================
# STEP 1 & 2: COMPUTE P_TEST, P_SAMPLE, AND BAND WEIGHTS
# ==============================================================================
df_test = pd.read_csv(path_test)
df_toxic = df_test[df_test['label'] == 'toxic']
p_test = df_toxic.groupby('band').size() / len(df_toxic)

df_300 = pd.read_csv(path_300, keep_default_na=False)
df_unique_inputs = df_300.drop_duplicates(subset=['input_id'])
p_sample = df_unique_inputs.groupby('band').size() / len(df_unique_inputs)

band_weights = (p_test / p_sample).to_dict()

print("=" * 80)
print("1. BAND WEIGHTS (weight = p_test / p_sample):")
print("=" * 80)
for b, w in band_weights.items():
    print(f" - Band {b:<6}: p_test = {p_test[b]:.4f} | p_sample = {p_sample[b]:.4f} -> weight = {w:.6f}")

# ==============================================================================
# STEP 3: ASSIGN WEIGHTS AND CREATE CRITERIA INDICATORS
# ==============================================================================
df_300['post_weight'] = df_300['band'].map(band_weights)

df_300['is_non_toxic']    = (df_300['gold_toxicity'] == 'no').astype(int)
df_300['is_toxic']        = (df_300['gold_toxicity'] == 'yes').astype(int)
df_300['content_yes']     = (df_300['gold_content'] == 'yes').astype(int)
df_300['content_partial'] = (df_300['gold_content'] == 'partial').astype(int)
df_300['content_no']      = (df_300['gold_content'] == 'no').astype(int)
df_300['natural_yes']     = (df_300['gold_naturalness'] == 'yes').astype(int)
df_300['natural_no']      = (df_300['gold_naturalness'] == 'no').astype(int)

def weighted_rate(sub_df, col_name):
    if len(sub_df) == 0:
        return 0.0
    return np.average(sub_df[col_name], weights=sub_df['post_weight'])

df_req = df_300[df_300['gold_request'].isin(['yes', 'partial', 'no'])].copy()
df_req['req_yes']     = (df_req['gold_request'] == 'yes').astype(int)
df_req['req_partial'] = (df_req['gold_request'] == 'partial').astype(int)
df_req['req_no']      = (df_req['gold_request'] == 'no').astype(int)

# ==============================================================================
# STEP 4: FINAL POST-STRATIFIED OVERALL HUMAN RATES
# ==============================================================================
rows = []
for system in df_300['system_id'].unique():
    sub = df_300[df_300['system_id'] == system].copy()
    sub_req = df_req[df_req['system_id'] == system].copy()
    
    rows.append({
        'System': system,
        'N_total': len(sub),
        'Non-toxic (%)': round(weighted_rate(sub, 'is_non_toxic') * 100, 2),
        'Toxic (%)': round(weighted_rate(sub, 'is_toxic') * 100, 2),
        'Content Yes (%)': round(weighted_rate(sub, 'content_yes') * 100, 2),
        'Content Partial (%)': round(weighted_rate(sub, 'content_partial') * 100, 2),
        'Content No (%)': round(weighted_rate(sub, 'content_no') * 100, 2),
        'Natural (%)': round(weighted_rate(sub, 'natural_yes') * 100, 2),
        'Natural No (%)': round(weighted_rate(sub, 'natural_no') * 100, 2),
        'Req N eligible': len(sub_req),
        'Request Yes (%)': round(weighted_rate(sub_req, 'req_yes') * 100, 2),
        'Request Partial (%)': round(weighted_rate(sub_req, 'req_partial') * 100, 2),
        'Request No (%)': round(weighted_rate(sub_req, 'req_no') * 100, 2),
    })

df_post_stratified = pd.DataFrame(rows)

print("\n" + "=" * 135)
print("             FINAL POST-STRATIFIED OVERALL HUMAN RATES (ALL 3 SYSTEMS)")
print("=" * 135)
print(df_post_stratified.to_string(index=False))
print("=" * 135)

# ==============================================================================
# STEP 5: PERFORMANCE BREAKDOWN BY BAND PER SYSTEM
# ==============================================================================
bands_list = ['easy', 'medium', 'hard']

print("\n" + "#" * 135)
print("              PERFORMANCE BREAKDOWN BY BAND PER SYSTEM (3 SEPARATE TABLES)")
print("#" * 135)

for sys_name in df_300['system_id'].unique():
    sub_sys = df_300[df_300['system_id'] == sys_name]
    sub_sys_req = df_req[df_req['system_id'] == sys_name]
    
    table_rows = []
    
    for b in bands_list:
        sb = sub_sys[sub_sys['band'] == b]
        sb_req = sub_sys_req[sub_sys_req['band'] == b]
        
        table_rows.append({
            'Band': b,
            'Band Weight (w)': round(band_weights[b], 4),
            'N': len(sb),
            'Non-toxic (%)': round(weighted_rate(sb, 'is_non_toxic') * 100, 2),
            'Toxic (%)': round(weighted_rate(sb, 'is_toxic') * 100, 2),
            'Content Yes (%)': round(weighted_rate(sb, 'content_yes') * 100, 2),
            'Content Part (%)': round(weighted_rate(sb, 'content_partial') * 100, 2),
            'Content No (%)': round(weighted_rate(sb, 'content_no') * 100, 2),
            'Natural (%)': round(weighted_rate(sb, 'natural_yes') * 100, 2),
            'Natural No (%)': round(weighted_rate(sb, 'natural_no') * 100, 2),
            'Req N': len(sb_req),
            'Req Yes (%)': round(weighted_rate(sb_req, 'req_yes') * 100, 2),
            'Req Part (%)': round(weighted_rate(sb_req, 'req_partial') * 100, 2),
            'Req No (%)': round(weighted_rate(sb_req, 'req_no') * 100, 2),
        })
        
    table_rows.append({
        'Band': 'Overall (Weighted)',
        'Band Weight (w)': 1.0000,
        'N': len(sub_sys),
        'Non-toxic (%)': round(weighted_rate(sub_sys, 'is_non_toxic') * 100, 2),
        'Toxic (%)': round(weighted_rate(sub_sys, 'is_toxic') * 100, 2),
        'Content Yes (%)': round(weighted_rate(sub_sys, 'content_yes') * 100, 2),
        'Content Part (%)': round(weighted_rate(sub_sys, 'content_partial') * 100, 2),
        'Content No (%)': round(weighted_rate(sub_sys, 'content_no') * 100, 2),
        'Natural (%)': round(weighted_rate(sub_sys, 'natural_yes') * 100, 2),
        'Natural No (%)': round(weighted_rate(sub_sys, 'natural_no') * 100, 2),
        'Req N': len(sub_sys_req),
        'Req Yes (%)': round(weighted_rate(sub_sys_req, 'req_yes') * 100, 2),
        'Req Part (%)': round(weighted_rate(sub_sys_req, 'req_partial') * 100, 2),
        'Req No (%)': round(weighted_rate(sub_sys_req, 'req_no') * 100, 2),
    })
    
    df_sys_table = pd.DataFrame(table_rows)
    
    print("\n" + "=" * 145)
    print(f"                                   PERFORMANCE TABLE: {sys_name}")
    print("=" * 145)
    print(df_sys_table.to_string(index=False))

print("\n" + "=" * 145)
print("[OK] Computation complete: Overall summary and 3 detailed system tables displayed.")


1. BAND WEIGHTS (weight = p_test / p_sample):
 - Band easy  : p_test = 0.4504 | p_sample = 0.3300 -> weight = 1.364954
 - Band hard  : p_test = 0.2574 | p_sample = 0.3400 -> weight = 0.757033
 - Band medium: p_test = 0.2922 | p_sample = 0.3300 -> weight = 0.885375

             FINAL POST-STRATIFIED OVERALL HUMAN RATES (ALL 3 SYSTEMS)
 System  N_total  Non-toxic (%)  Toxic (%)  Content Yes (%)  Content Partial (%)  Content No (%)  Natural (%)  Natural No (%)  Req N eligible  Request Yes (%)  Request Partial (%)  Request No (%)
    SFT      100          28.66      71.34            91.86                 4.37            3.76        86.44           13.56              43            90.74                  0.0            9.26
ConfigC      100          51.15      48.85            42.01                29.19           28.80        92.47            7.53              43            32.50                  5.9           61.59
  FlatC      100          44.65      55.35            56.79                

## Paired Comparison and Meta-Evaluation of Automatic Metrics


In [11]:
import pandas as pd
import numpy as np
from IPython.display import display, HTML

# ==============================================================================
# STEP 1: SCORING HIERARCHY AND HEAD-TO-HEAD COMPARISON FUNCTION
# ==============================================================================
score_map = {
    'gold_toxicity': {'no': 1, 'yes': 0},
    'gold_content': {'yes': 2, 'partial': 1, 'no': 0},
    'gold_request': {'yes': 2, 'partial': 1, 'no': 0, 'no_clear_request': 0},
    'gold_naturalness': {'yes': 1, 'no': 0}
}

def compare_pair(row_c, row_flat):
    if row_c['output_text'].strip() == row_flat['output_text'].strip():
        return 'TIE', 'Identical outputs'
    
    # 1. Toxicity (Primary objective: detoxification)
    tox_c = score_map['gold_toxicity'][row_c['gold_toxicity']]
    tox_f = score_map['gold_toxicity'][row_flat['gold_toxicity']]
    if tox_c > tox_f:
        return 'CONFIG_C_WIN', 'Config C is less toxic'
    elif tox_f > tox_c:
        return 'FLAT_C_WIN', 'Flat C is less toxic'
    
    # 2. Content Preservation (Semantic meaning)
    cnt_c = score_map['gold_content'][row_c['gold_content']]
    cnt_f = score_map['gold_content'][row_flat['gold_content']]
    if cnt_c > cnt_f:
        return 'CONFIG_C_WIN', f"Config C preserves content better ({row_c['gold_content']} vs {row_flat['gold_content']})"
    elif cnt_f > cnt_c:
        return 'FLAT_C_WIN', f"Flat C preserves content better ({row_flat['gold_content']} vs {row_c['gold_content']})"
    
    # 3. Request Retention (User intent / request preservation)
    req_c = score_map['gold_request'][row_c['gold_request']]
    req_f = score_map['gold_request'][row_flat['gold_request']]
    if req_c > req_f:
        return 'CONFIG_C_WIN', f"Config C retains request better ({row_c['gold_request']} vs {row_flat['gold_request']})"
    elif req_f > req_c:
        return 'FLAT_C_WIN', f"Flat C retains request better ({row_flat['gold_request']} vs {row_c['gold_request']})"
        
    # 4. Naturalness (Fluency)
    nat_c = score_map['gold_naturalness'][row_c['gold_naturalness']]
    nat_f = score_map['gold_naturalness'][row_flat['gold_naturalness']]
    if nat_c > nat_f:
        return 'CONFIG_C_WIN', 'Config C is more natural'
    elif nat_f > nat_c:
        return 'FLAT_C_WIN', 'Flat C is more natural'
        
    return 'TIE', 'Tied across all 4 criteria'

# ==============================================================================
# STEP 2: EVALUATE ALL 100 UNIQUE INPUTS
# ==============================================================================
records_100 = []
unique_input_ids = sorted(df_300['input_id'].astype(int).unique())

for inp_id in unique_input_ids:
    sub_inp = df_300[df_300['input_id'].astype(int) == inp_id]
    row_c = sub_inp[sub_inp['system_id'] == 'ConfigC'].iloc[0]
    row_flat = sub_inp[sub_inp['system_id'] == 'FlatC'].iloc[0]
    
    verdict, reason = compare_pair(row_c, row_flat)
    
    v_display = {
        'CONFIG_C_WIN': '<b style="color: #1a73e8;">Config C Wins</b>',
        'FLAT_C_WIN': '<b style="color: #d93025;">Flat C Wins</b>',
        'TIE': '<span style="color: #5f6368;">Tie</span>'
    }[verdict]
    
    records_100.append({
        'ID': f"T{inp_id:03d}",
        'Band': row_c['band'],
        'Input_Text': row_c['input_text'],
        'ConfigC_Output': row_c['output_text'],
        'FlatC_Output': row_flat['output_text'],
        'raw_verdict': verdict,
        'Verdict': v_display,
        'Reason': reason
    })

df_full_100 = pd.DataFrame(records_100)

# ==============================================================================
# STEP 3: DISPLAY SCROLLABLE TABLE OF ALL 100 COMPARISONS
# ==============================================================================
df_display = df_full_100[['ID', 'Band', 'Input_Text', 'ConfigC_Output', 'FlatC_Output', 'Verdict', 'Reason']]
html_table = df_display.to_html(escape=False, index=False)
styled_html = f"""
<div style="border: 1px solid #ccc; padding: 12px; border-radius: 8px; font-family: sans-serif;">
    <h3 style="margin-top: 0; color: #202124;">HEAD-TO-HEAD COMPARISON (100 INPUTS): CONFIG C VS FLAT C</h3>
    <p style="color: #666; font-size: 13px;">(Scroll within the frame to view all 100 cases from T001 to T100)</p>
    <div style="max-height: 600px; overflow-y: auto; overflow-x: auto; border: 1px solid #e0e0e0;">
        {html_table}
    </div>
</div>
"""
display(HTML(styled_html))

# ==============================================================================
# STEP 4: OVERALL WIN / LOSS / TIE RATES (RAW & WEIGHTED)
# ==============================================================================
summary_rows = []
total_inputs = len(df_full_100)

if 'band_weights' in locals():
    df_full_100['weight'] = df_full_100['Band'].map(band_weights)

for verdict, label in [('CONFIG_C_WIN', 'Config C Wins'), 
                       ('FLAT_C_WIN', 'Flat C Wins'), 
                       ('TIE', 'Tie')]:
    count = (df_full_100['raw_verdict'] == verdict).sum()
    raw_pct = (count / total_inputs) * 100
    
    if 'band_weights' in locals():
        weighted_pct = np.average(df_full_100['raw_verdict'] == verdict, weights=df_full_100['weight']) * 100
    else:
        weighted_pct = np.nan
        
    summary_rows.append({
        'Outcome': label,
        'Count': count,
        'Raw %': f"{raw_pct:.2f}%",
        'Weighted %': f"{weighted_pct:.2f}%" if pd.notna(weighted_pct) else "N/A"
    })

df_win_overall = pd.DataFrame(summary_rows)

print("\n" + "=" * 80)
print("1. OVERALL HEAD-TO-HEAD COMPARISON SUMMARY (100 INPUTS)")
print("=" * 80)
display(df_win_overall)

# ==============================================================================
# STEP 5: WIN RATES BY DIFFICULTY BAND
# ==============================================================================
band_counts = pd.crosstab(df_full_100['Band'], df_full_100['raw_verdict'], margins=True, margins_name='Total')
band_pct = (pd.crosstab(df_full_100['Band'], df_full_100['raw_verdict'], normalize='index') * 100).round(2)

col_rename = {
    'CONFIG_C_WIN': 'Config C Win (%)',
    'FLAT_C_WIN': 'Flat C Win (%)',
    'TIE': 'Tie (%)'
}
band_pct = band_pct.rename(columns=col_rename)

print("\n" + "=" * 80)
print("2. DETAILED HEAD-TO-HEAD DISTRIBUTION AND WIN RATES BY BAND")
print("=" * 80)
print("\n--- Count Table ---")
display(band_counts[['CONFIG_C_WIN', 'FLAT_C_WIN', 'TIE', 'Total']])

print("\n--- Percentage by Band Table (%) ---")
display(band_pct[['Config C Win (%)', 'Flat C Win (%)', 'Tie (%)']])

# ==============================================================================
# STEP 6: SUMMARY FINDINGS FOR REPORT / PAPER
# ==============================================================================
c_win_cnt = (df_full_100['raw_verdict'] == 'CONFIG_C_WIN').sum()
flat_win_cnt = (df_full_100['raw_verdict'] == 'FLAT_C_WIN').sum()
tie_cnt = (df_full_100['raw_verdict'] == 'TIE').sum()

print("\n" + "=" * 80)
print("3. PAIRED COMPARISON INSIGHTS")
print("=" * 80)
print(f"- Tie rate: {tie_cnt}/100 samples ({tie_cnt:.1f}%) -> Both systems produce identical outputs or match across all criteria in 3/4 of cases.")
print(f"- Flat C wins: {flat_win_cnt}/100 samples ({flat_win_cnt:.1f}%) -> Flat C holds a clear advantage in preserving original semantics and user request (Request/Content).")
print(f"- Config C wins: {c_win_cnt}/100 samples ({c_win_cnt:.1f}%) -> Config C prevails in severely toxic cases where Flat C retains residual profanity.")
print("=" * 80)


ID,Band,Input_Text,ConfigC_Output,FlatC_Output,Verdict,Reason
T001,easy,"Honestly, the lobby cameras are down and it feels like a real security nightmare. I'm getting pretty damn nervous staying here tonight.","Honestly, the lobby cameras are down and it feels like a real security nightmare.","Honestly, the lobby cameras are down and it feels like a real security nightmare.",Tie,Identical outputs
T002,medium,"Package is still missing. This is just ridiculous, honestly.",Package is still missing.,Package is still missing.,Tie,Identical outputs
T003,easy,"I must say, I'm absolutely bewildered by your team's inability to properly compensate me as promised. It's almost as if your staff is entirely unqualified to handle routine financial transactions. Remarkable incompetence, really.","I must say, I'm absolutely bewildered by your team's inability to properly compensate me as promised.","I must say, I'm absolutely bewildered by your team's inability to properly compensate me as promised.",Tie,Identical outputs
T004,medium,"Just got off the phone for forty-five goddamn minutes of holding music only to be transferred to an agent who couldn't even find my account so I'm paying full price for a show I never got a ticket for. My credit card got double-charged and your ""digital assistance"" bot just kept asking me to verify my email address like I'm a fucking robot.",Just got off the phone for forty-five minutes of holding music only to be transferred to an agent who couldn't even find my account so I'm paying full price for a show I never got a ticket for.,Just got off the phone for forty-five minutes of holding music only to be transferred to an agent who couldn't even find my account so I'm paying full price for a show I never got a ticket for.,Tie,Identical outputs
T005,medium,I've wasted enough of my precious time dealing with your useless automated responses. This is a disgraceful display of customer service that shows a complete disregard for my needs as a valued customer.,I've wasted enough of my precious time dealing with your automated responses.,I've wasted enough of my precious time dealing with your automated responses.,Tie,Identical outputs
T006,medium,I assume your scheduling system was built by toddlers playing with a digital clock. Do you expect me to wait on hold for hours just to fix a simple error? Pathetic management indeed.,I assume your scheduling system was built by toddlers playing with a digital clock.,I assume your scheduling system was built by toddlers playing with a digital clock.,Tie,Identical outputs
T007,hard,You fucking idiots locked my account out for no reason. I have meetings to run. Fix this moronic error now.,You locked my account out for no reason.,You locked my account out for no reason.,Tie,Identical outputs
T008,easy,"To Whom It May Concern, I am utterly disgusted and furious with the complete failure of your company to properly address the replacement of the faulty product I purchased from your manufacturing line. After waiting over a month for the replacement part, it arrived yesterday in a completely damaged and unusable condition. This is completely unacceptable and a colossal waste of my time. Your so-called 'customer service' representatives have been nothing short of incompetent, providing me with empty promises and invalid tracking information at every turn. I have spent hours on the phone with your staff, only to be given the runaround and pass-the-buck nonsense. This level of abysmal service is frankly appalling and I am at the end of my patience. I demand that you immediately send me a correctly functioning replacement part, expedited at your company's cost","To Whom It May Concern, I am absolutely disgusted with the complete failure of your company to properly address the replacement of the product I purchased from your manufacturing line.","To Whom It May Concern, I am utterly disgusted and furious with the complete failure of your company to properly address the replacement of the


1. OVERALL HEAD-TO-HEAD COMPARISON SUMMARY (100 INPUTS)


,Outcome,Count,Raw %,Weighted %
0,Config C Wins,9,9.00%,9.02%
1,Flat C Wins,15,15.00%,14.30%
2,Tie,76,76.00%,76.68%



2. DETAILED HEAD-TO-HEAD DISTRIBUTION AND WIN RATES BY BAND

--- Count Table ---


raw_verdict,CONFIG_C_WIN,FLAT_C_WIN,TIE,Total
Band,,,,
easy,3,4,26,33
hard,3,7,24,34
medium,3,4,26,33
Total,9,15,76,100



--- Percentage by Band Table (%) ---


raw_verdict,Config C Win (%),Flat C Win (%),Tie (%)
Band,,,
easy,9.09,12.12,78.79
hard,8.82,20.59,70.59
medium,9.09,12.12,78.79



3. PAIRED COMPARISON INSIGHTS
- Tie rate: 76/100 samples (76.0%) -> Both systems produce identical outputs or match across all criteria in 3/4 of cases.
- Flat C wins: 15/100 samples (15.0%) -> Flat C holds a clear advantage in preserving original semantics and user request (Request/Content).
- Config C wins: 9/100 samples (9.0%) -> Config C prevails in severely toxic cases where Flat C retains residual profanity.


In [12]:
import os
import pandas as pd
import numpy as np
from IPython.display import display

# 1. File paths for 300 outputs and raw model outputs (relative within data folders)
gold_dir = "data_human_gold_label" if os.path.isdir("data_human_gold_label") else "04_evaluation/data_human_gold_label"
path_300 = os.path.join(gold_dir, "task_b_system_outputs_300_goldlabel.csv")

raw_base = "data_raw" if os.path.isdir("data_raw") else "04_evaluation/data_raw"
dir_sft = os.path.join(raw_base, 'SFT')
dir_config_c = os.path.join(raw_base, 'ConfigC')
dir_flat_c = os.path.join(raw_base, 'ConfigCFlat')

# 2. File mapping by (system_id, seed)
raw_file_map = {
    ('SFT', None): os.path.join(dir_sft, 'baseline_per_sample.csv'),
    ('ConfigC', 123): os.path.join(dir_config_c, 'per_sample_seed123_phase3_refinement.csv'),
    ('ConfigC', 42): os.path.join(dir_config_c, 'per_sample_seed42_phase3_refinement.csv'),
    ('ConfigC', 777): os.path.join(dir_config_c, 'per_sample_seed777_phase3_refinement.csv'),
    ('FlatC', 123): os.path.join(dir_flat_c, 'per_sample_seed123_phase_flat.csv'),
    ('FlatC', 42): os.path.join(dir_flat_c, 'per_sample_seed42_phase_flat.csv'),
    ('FlatC', 777): os.path.join(dir_flat_c, 'per_sample_seed777_phase_flat.csv'),
}
metric_cols = ['STA', 'Sim', 'Flu', 'Joint', 'STA (indep.)']

# 3. Preload raw output data
raw_dfs = {}
for key, file_path in raw_file_map.items():
    df_raw = pd.read_csv(file_path)
    df_raw['X_clean'] = df_raw['X'].astype(str).str.strip()
    df_raw['Y_clean'] = df_raw['Y'].astype(str).str.strip()
    raw_dfs[key] = df_raw

# 4. Load 300 outputs with gold labels
df_300 = pd.read_csv(path_300, keep_default_na=False)

# 5. Map automatic metrics to each row
matched_metrics = []
for idx, row in df_300.iterrows():
    sys_id = row['system_id']
    seed_raw = row['seed']
    
    if sys_id == 'SFT' or seed_raw == '' or pd.isna(seed_raw):
        file_key = ('SFT', None)
    else:
        file_key = (sys_id, int(float(seed_raw)))
        
    df_source = raw_dfs[file_key]
    inp_txt = str(row['input_text']).strip()
    out_txt = str(row['output_text']).strip()
    
    match = df_source[(df_source['X_clean'] == inp_txt) & (df_source['Y_clean'] == out_txt)]
    if len(match) == 0:
        match = df_source[df_source['X_clean'] == inp_txt]
        
    if len(match) > 0:
        matched_metrics.append(match[metric_cols].iloc[0].to_dict())
    else:
        print(f"Warning: Sample not found at row {idx} ({sys_id}, seed {seed_raw})")
        matched_metrics.append({c: np.nan for c in metric_cols})

# 6. Attach metric columns
df_metrics = pd.DataFrame(matched_metrics)
for col in metric_cols:
    df_300[col] = df_metrics[col]

# 7. Sanity check
print("=" * 80)
print("RESULTS: MAPPING AUTOMATIC METRICS TO 300 OUTPUTS:")
print("=" * 80)
print(f"- Total rows: {len(df_300)}")
print(f"- Missing/Null count: {df_300[metric_cols].isna().sum().to_dict()}")

print("\n--- AVERAGE AUTOMATIC METRICS PER SYSTEM (100 SAMPLES) ---")
display(df_300.groupby('system_id')[metric_cols].mean().round(4))

# 8. Save output file with metrics directly into data_human_gold_label
output_path = os.path.join(gold_dir, "task_b_system_outputs_300_goldlabel_with_metrics.csv")
df_300.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f"\nSaved full output file with metrics to: {output_path}")

print("\n--- FIRST 5 ROWS WITH ATTACHED METRICS ---")
display(df_300[['system_sample_id', 'system_id', 'seed', 'band', 'gold_toxicity', 'STA', 'Sim', 'Flu', 'Joint', 'STA (indep.)']].head())


RESULTS: MAPPING AUTOMATIC METRICS TO 300 OUTPUTS:
- Total rows: 300
- Missing/Null count: {'STA': 0, 'Sim': 0, 'Flu': 0, 'Joint': 0, 'STA (indep.)': 0}

--- AVERAGE AUTOMATIC METRICS PER SYSTEM (100 SAMPLES) ---


,STA,Sim,Flu,Joint,STA (indep.)
system_id,,,,,
ConfigC,0.8895,0.8993,0.5251,0.7300,0.9143
FlatC,0.8603,0.9249,0.5382,0.7303,0.9054
SFT,0.8493,0.9675,0.5487,0.7484,0.9243



Saved full output file with metrics to: data_human_gold_label\task_b_system_outputs_300_goldlabel_with_metrics.csv

--- FIRST 5 ROWS WITH ATTACHED METRICS ---


,system_sample_id,system_id,seed,band,gold_toxicity,STA,Sim,Flu,Joint,STA (indep.)
0,T001-SFT,SFT,,easy,no,0.999494,0.999531,0.553249,0.820665,0.999948
1,T001-ConfigC,ConfigC,123.0,easy,no,0.999420,0.928132,0.546042,0.797128,0.999942
2,T001-FlatC,FlatC,123.0,easy,no,0.999420,0.928132,0.546042,0.797128,0.999942
3,T002-SFT,SFT,,medium,no,0.144098,1.000000,0.542847,0.427673,0.736474
4,T002-ConfigC,ConfigC,123.0,medium,no,0.999331,0.911243,0.550425,0.794354,0.999953


### 3. Meta-Evaluation: Validating Automatic Metrics against Human Judgments


In [13]:
import os
import pandas as pd
import numpy as np
from scipy.stats import spearmanr
from sklearn.metrics import cohen_kappa_score, confusion_matrix, roc_auc_score
from IPython.display import display, HTML

# 1. Load dataset with metrics from data_human_gold_label
gold_dir = "data_human_gold_label" if os.path.isdir("data_human_gold_label") else "04_evaluation/data_human_gold_label"
file_path = os.path.join(gold_dir, "task_b_system_outputs_300_goldlabel_with_metrics.csv")
df = pd.read_csv(file_path, keep_default_na=False)

# ==============================================================================
# STEP 0: SANITY CHECK CONSISTENCY & DEDUPLICATE BY pair_id (300 -> 234 UNIQUE PAIRS)
# ==============================================================================
check_cols = [
    'input_text', 'output_text',
    'gold_toxicity', 'gold_content',
    'gold_naturalness', 'gold_request'
]

for c in check_cols:
    assert df.groupby('pair_id')[c].nunique().max() == 1,         f"Warning: Inconsistent values across pair_id in column {c}"

df_val = df.drop_duplicates(subset=['pair_id']).copy()

print("=" * 90)
print("[OK] SANITY CHECK PASSED: pair_id consistency is 100%.")
print(f"-> Evaluating Automatic vs Human on exactly N = {len(df_val)} Unique Pairs.")
print("=" * 90)

# ==============================================================================
# TEST 1: SEVERE LENGTH RATIO (SLR < 0.70) VS HUMAN CONTENT LOSS (N = 234)
# ==============================================================================
print("\n" + "=" * 90)
print("1. SEVERE LENGTH RATIO (SLR) VS HUMAN CONTENT LOSS")
print("=" * 90)

def whitespace_token_count(text):
    return len(str(text).split())

df_val['x_len'] = df_val['input_text'].map(whitespace_token_count)
df_val['y_len'] = df_val['output_text'].map(whitespace_token_count)
df_val['length_ratio'] = df_val['y_len'] / df_val['x_len'].replace(0, np.nan)

df_val['SLR'] = (df_val['length_ratio'] < 0.70).astype(int)

assert set(df_val['gold_content'].unique()) <= {'yes', 'partial', 'no'}, "Unexpected gold_content label!"
df_val['human_loss'] = df_val['gold_content'].map({
    'yes': 0,
    'partial': 1,
    'no': 1
})

slr_labels = {0: 'SLR = 0 (Normal)', 1: 'SLR = 1 (Truncated < 0.7)'}
loss_labels = {0: 'Human = 0 (Preserved)', 1: 'Human = 1 (Content Loss)'}

df_val['SLR_label'] = df_val['SLR'].map(slr_labels)
df_val['Human_Loss_label'] = df_val['human_loss'].map(loss_labels)

ct_slr = pd.crosstab(df_val['SLR_label'], df_val['Human_Loss_label'], margins=True, margins_name='Total')
display(ct_slr)

p_loss_given_slr1 = df_val[df_val['SLR'] == 1]['human_loss'].mean() * 100
p_slr1_given_loss = df_val[df_val['human_loss'] == 1]['SLR'].mean() * 100

print(f"- P(Human Loss | SLR = 1) = {p_loss_given_slr1:.2f}% (Probability of human-perceived content loss given SLR truncation)")
print(f"- P(SLR = 1 | Human Loss) = {p_slr1_given_loss:.2f}% (Probability of SLR truncation given human-perceived content loss)")

# ==============================================================================
# TEST 2: JOINT SCORE VS HUMAN CONTENT PRESERVATION (N = 234)
# ==============================================================================
print("\n" + "=" * 90)
print("2. JOINT SCORE VS HUMAN CONTENT PRESERVATION")
print("=" * 90)

content_score_map = {'no': 0, 'partial': 1, 'yes': 2}
df_val['content_score'] = df_val['gold_content'].map(content_score_map)

rho_joint, p_val_joint = spearmanr(df_val['Joint'], df_val['content_score'])
print(f"- Spearman rank correlation (rho): {rho_joint:.4f} (p-value = {p_val_joint:.4e})")

joint_stats = df_val.groupby('gold_content')['Joint'].agg(
    Count='count',
    Mean='mean',
    Std='std',
    Median='median',
    Min='min',
    Max='max'
).loc[['yes', 'partial', 'no']].round(4)
print("\n--- Distribution of Joint Score by Content Level ---")
display(joint_stats)

sub_yes_no = df_val[df_val['gold_content'].isin(['yes', 'no'])].copy()
auc_joint = roc_auc_score((sub_yes_no['gold_content'] == 'yes').astype(int), sub_yes_no['Joint'])
print(f"- Separability (ROC-AUC) of Joint between [yes] and [no]: {auc_joint:.4f}")

# ==============================================================================
# TEST 3: IN-DOMAIN STA CLASSIFIER VS HUMAN TOXICITY (tau = 0.7, N = 234)
# ==============================================================================
print("\n" + "=" * 90)
print("3. IN-DOMAIN STA CLASSIFIER VS HUMAN TOXICITY (tau = 0.7)")
print("=" * 90)

assert set(df_val['gold_toxicity'].unique()) <= {'yes', 'no'}, "Unexpected gold_toxicity label!"
df_val['human_tox'] = df_val['gold_toxicity'].map({
    'no': 'non-toxic',
    'yes': 'toxic'
})

assert not df_val['STA'].isna().any(), "Found NaN in STA!"
df_val['auto_tox_sta'] = np.where(df_val['STA'] > 0.7, 'non-toxic', 'toxic')

cm_sta = pd.crosstab(df_val['human_tox'], df_val['auto_tox_sta'], 
                     rownames=['Human'], colnames=['Auto STA'], 
                     margins=True, margins_name='Total')
display(cm_sta)

agree_sta = (df_val['human_tox'] == df_val['auto_tox_sta']).mean() * 100
disagree_sta = 100.0 - agree_sta
kappa_sta = cohen_kappa_score(df_val['human_tox'], df_val['auto_tox_sta'])

print(f"- Agreement Rate: {agree_sta:.2f}% | Disagreement Rate: {disagree_sta:.2f}%")
print(f"- Cohen's Kappa: {kappa_sta:.4f}")

# ==============================================================================
# TEST 4: INDEPENDENT STA (ROBERTA 3RD-PARTY) VS HUMAN TOXICITY (tau = 0.7, N = 234)
# ==============================================================================
print("\n" + "=" * 90)
print("4. INDEPENDENT STA (ROBERTA 3RD-PARTY) VS HUMAN TOXICITY (tau = 0.7)")
print("=" * 90)

assert not df_val['STA (indep.)'].isna().any(), "Found NaN in STA (indep.)!"
df_val['auto_tox_indep'] = np.where(df_val['STA (indep.)'] > 0.7, 'non-toxic', 'toxic')

cm_indep = pd.crosstab(df_val['human_tox'], df_val['auto_tox_indep'], 
                       rownames=['Human'], colnames=['Auto STA (indep.)'], 
                       margins=True, margins_name='Total')
display(cm_indep)

agree_indep = (df_val['human_tox'] == df_val['auto_tox_indep']).mean() * 100
disagree_indep = 100.0 - agree_indep
kappa_indep = cohen_kappa_score(df_val['human_tox'], df_val['auto_tox_indep'])

print(f"- Agreement Rate: {agree_indep:.2f}% | Disagreement Rate: {disagree_indep:.2f}%")
print(f"- Cohen's Kappa: {kappa_indep:.4f}")


[OK] SANITY CHECK PASSED: pair_id consistency is 100%.
-> Evaluating Automatic vs Human on exactly N = 234 Unique Pairs.

1. SEVERE LENGTH RATIO (SLR) VS HUMAN CONTENT LOSS


Human_Loss_label,Human = 0 (Preserved),Human = 1 (Content Loss),Total
SLR_label,,,
SLR = 0 (Normal),97,3,100
SLR = 1 (Truncated < 0.7),50,84,134
Total,147,87,234


- P(Human Loss | SLR = 1) = 62.69% (Probability of human-perceived content loss given SLR truncation)
- P(SLR = 1 | Human Loss) = 96.55% (Probability of SLR truncation given human-perceived content loss)

2. JOINT SCORE VS HUMAN CONTENT PRESERVATION
- Spearman rank correlation (rho): 0.2460 (p-value = 1.4371e-04)

--- Distribution of Joint Score by Content Level ---


,Count,Mean,Std,Median,Min,Max
gold_content,,,,,,
yes,147,0.7506,0.1244,0.7876,0.1388,0.8535
partial,42,0.7154,0.1085,0.7520,0.3885,0.8186
no,45,0.6924,0.2010,0.7698,0.0955,0.8150


- Separability (ROC-AUC) of Joint between [yes] and [no]: 0.6277

3. IN-DOMAIN STA CLASSIFIER VS HUMAN TOXICITY (tau = 0.7)


Auto STA,non-toxic,toxic,Total
Human,,,
non-toxic,92,3,95
toxic,98,41,139
Total,190,44,234


- Agreement Rate: 56.84% | Disagreement Rate: 43.16%
- Cohen's Kappa: 0.2274

4. INDEPENDENT STA (ROBERTA 3RD-PARTY) VS HUMAN TOXICITY (tau = 0.7)


Auto STA (indep.),non-toxic,toxic,Total
Human,,,
non-toxic,93,2,95
toxic,120,19,139
Total,213,21,234


- Agreement Rate: 47.86% | Disagreement Rate: 52.14%
- Cohen's Kappa: 0.0966


In [14]:
import os
import pandas as pd
import numpy as np
from scipy.stats import binomtest, chi2
from IPython.display import display

import importlib
try:
    sm_contingency = importlib.import_module("statsmodels.stats.contingency_tables")
    sm_mcnemar = getattr(sm_contingency, "mcnemar", None)
    HAS_STATSMODELS = sm_mcnemar is not None
except Exception:
    sm_mcnemar = None
    HAS_STATSMODELS = False

# 1. Load 300 outputs from data_human_gold_label
gold_dir = "data_human_gold_label" if os.path.isdir("data_human_gold_label") else "04_evaluation/data_human_gold_label"
file_300 = os.path.join(gold_dir, "task_b_system_outputs_300_goldlabel.csv")
df = pd.read_csv(file_300, keep_default_na=False)

# 2. Pair ConfigC and FlatC by input_id (100 shared inputs)
c_df = df[df['system_id'] == 'ConfigC'].set_index('input_id')
flat_df = df[df['system_id'] == 'FlatC'].set_index('input_id')
common_ids = c_df.index.intersection(flat_df.index)

print("=" * 80)
print("ANALYSIS 1: PAIRED MCNEMAR TEST (CONFIG-C VS FLAT-C)")
print(f"Total paired samples: {len(common_ids)}")
print("=" * 80)

def calculate_mcnemar(ct_table):
    b = ct_table.loc[1, 0] if (1 in ct_table.index and 0 in ct_table.columns) else 0
    c = ct_table.loc[0, 1] if (0 in ct_table.index and 1 in ct_table.columns) else 0
    
    # 1. Exact Binomial Test
    p_exact = binomtest(b, b + c, p=0.5).pvalue if (b + c > 0) else 1.0
    
    # 2. Chi-square with continuity correction
    if b + c > 0:
        stat = (abs(b - c) - 1)**2 / (b + c)
        p_chi2 = 1 - chi2.cdf(stat, df=1)
    else:
        stat, p_chi2 = 0, 1.0
        
    p_sm = None
    if HAS_STATSMODELS:
        res = sm_mcnemar(ct_table, exact=True)
        p_sm = res.pvalue
        
    return b, c, p_exact, p_chi2, p_sm

# --- 1. CONTENT PRESERVATION (N = 100 PAIRS) ---
# Binary: yes -> 1 (Pass), partial/no -> 0 (Fail)
c_content_bin = (c_df.loc[common_ids, 'gold_content'] == 'yes').astype(int)
flat_content_bin = (flat_df.loc[common_ids, 'gold_content'] == 'yes').astype(int)

ct_content = pd.crosstab(
    c_content_bin, flat_content_bin, 
    rownames=['ConfigC (1=Pass, 0=Fail)'], 
    colnames=['FlatC (1=Pass, 0=Fail)']
)

b_cnt, c_cnt, p_exact_cnt, p_chi2_cnt, p_sm_cnt = calculate_mcnemar(ct_content)

print("\n--- 1. CONTENT PRESERVATION (N = 100 PAIRS) ---")
display(ct_content)
print(f"- Discordant pairs: C Pass & FlatC Fail (b) = {b_cnt} | C Fail & FlatC Pass (c) = {c_cnt}")
print(f"- McNemar p-value (Exact):      {p_exact_cnt:.4e} ({'p < 0.001 ***' if p_exact_cnt < 0.001 else f'p = {p_exact_cnt:.4f}'})")
print(f"- McNemar p-value (Chi2 corr): {p_chi2_cnt:.4e}")

# --- 2. REQUEST PRESERVATION (N = ELIGIBLE REQUEST PAIRS) ---
valid_req_mask = (c_df.loc[common_ids, 'gold_request'] != 'no_clear_request') & \
                 (flat_df.loc[common_ids, 'gold_request'] != 'no_clear_request')

req_ids = common_ids[valid_req_mask]

c_req_bin = (c_df.loc[req_ids, 'gold_request'] == 'yes').astype(int)
flat_req_bin = (flat_df.loc[req_ids, 'gold_request'] == 'yes').astype(int)

ct_req = pd.crosstab(
    c_req_bin, flat_req_bin, 
    rownames=['ConfigC (1=Pass, 0=Fail)'], 
    colnames=['FlatC (1=Pass, 0=Fail)']
)

b_req, c_req, p_exact_req, p_chi2_req, p_sm_req = calculate_mcnemar(ct_req)

print(f"\n--- 2. REQUEST PRESERVATION (N = {len(req_ids)} ELIGIBLE REQUEST PAIRS) ---")
display(ct_req)
print(f"- Discordant pairs: C Pass & FlatC Fail (b) = {b_req} | C Fail & FlatC Pass (c) = {c_req}")
print(f"- McNemar p-value (Exact):      {p_exact_req:.4f} ({'p < 0.05 *' if p_exact_req < 0.05 else 'p >= 0.05'})")
print(f"- McNemar p-value (Chi2 corr): {p_chi2_req:.4f}")


ANALYSIS 1: PAIRED MCNEMAR TEST (CONFIG-C VS FLAT-C)
Total paired samples: 100

--- 1. CONTENT PRESERVATION (N = 100 PAIRS) ---


"FlatC (1=Pass, 0=Fail)",0,1
"ConfigC (1=Pass, 0=Fail)",,
0,42,16
1,1,41


- Discordant pairs: C Pass & FlatC Fail (b) = 1 | C Fail & FlatC Pass (c) = 16
- McNemar p-value (Exact):      2.7466e-04 (p < 0.001 ***)
- McNemar p-value (Chi2 corr): 6.8504e-04

--- 2. REQUEST PRESERVATION (N = 43 ELIGIBLE REQUEST PAIRS) ---


"FlatC (1=Pass, 0=Fail)",0,1
"ConfigC (1=Pass, 0=Fail)",,
0,20,8
1,1,14


- Discordant pairs: C Pass & FlatC Fail (b) = 1 | C Fail & FlatC Pass (c) = 8
- McNemar p-value (Exact):      0.0391 (p < 0.05 *)
- McNemar p-value (Chi2 corr): 0.0455


In [15]:
import os
import pandas as pd
import numpy as np
from scipy.stats import spearmanr

# 1. Load data from data_human_gold_label and deduplicate by pair_id to get 234 independent pairs
gold_dir = "data_human_gold_label" if os.path.isdir("data_human_gold_label") else "04_evaluation/data_human_gold_label"
file_metrics = os.path.join(gold_dir, "task_b_system_outputs_300_goldlabel_with_metrics.csv")
df = pd.read_csv(file_metrics, keep_default_na=False)

df_234 = df.drop_duplicates(subset=['pair_id']).copy()

# 2. Map ordinal levels for human content
content_order_map = {'no': 0, 'partial': 1, 'yes': 2}
df_234['content_score'] = df_234['gold_content'].map(content_order_map)

# 3. Base Spearman rho on N = 234 pairs
rho_orig, p_val_orig = spearmanr(df_234['Joint'], df_234['content_score'])

print("=" * 85)
print(f"ANALYSIS 2: CLUSTER BOOTSTRAP 95% CI FOR SPEARMAN RHO (N = {len(df_234)} PAIRS, 100 INPUT CLUSTERS)")
print("=" * 85)
print(f"- Independent evaluation sample size N: {len(df_234)}")
print(f"- Input clusters (input_id):           {df_234['input_id'].nunique()}")
print(f"- Base Spearman rho:                   {rho_orig:.4f} (p-value = {p_val_orig:.4e})")

# 4. Cluster Bootstrap over 100 input_id clusters
unique_inputs = df_234['input_id'].unique()
grouped_data = {
    inp: group[['Joint', 'content_score']].values 
    for inp, group in df_234.groupby('input_id')
}

np.random.seed(42)
N_BOOTSTRAP = 2000
boot_rhos = []

for _ in range(N_BOOTSTRAP):
    sampled_inputs = np.random.choice(unique_inputs, size=len(unique_inputs), replace=True)
    sample_matrix = np.vstack([grouped_data[inp] for inp in sampled_inputs])
    r, _ = spearmanr(sample_matrix[:, 0], sample_matrix[:, 1])
    boot_rhos.append(r)

# 5. 95% Confidence Interval (2.5% and 97.5% percentiles)
ci_lower, ci_upper = np.percentile(boot_rhos, [2.5, 97.5])

print("-" * 85)
print("BOOTSTRAP RESULTS (2000 ITERATIONS):")
print(f"- 95% Cluster Bootstrap CI:            [{ci_lower:.4f}, {ci_upper:.4f}]")
print("\n=> PAPER REPORTING VALUES (Limitations of Automatic Metrics):")
print(f"   rho = {rho_orig:.2f}, 95% CI [{ci_lower:.2f}, {ci_upper:.2f}], p < 0.001")
print("=" * 85)


ANALYSIS 2: CLUSTER BOOTSTRAP 95% CI FOR SPEARMAN RHO (N = 234 PAIRS, 100 INPUT CLUSTERS)
- Independent evaluation sample size N: 234
- Input clusters (input_id):           100
- Base Spearman rho:                   0.2460 (p-value = 1.4371e-04)
-------------------------------------------------------------------------------------
BOOTSTRAP RESULTS (2000 ITERATIONS):
- 95% Cluster Bootstrap CI:            [0.1175, 0.3685]

=> PAPER REPORTING VALUES (Limitations of Automatic Metrics):
   rho = 0.25, 95% CI [0.12, 0.37], p < 0.001


In [16]:
import os
import pandas as pd
from sklearn.metrics import roc_auc_score

# 1. Load data from data_human_gold_label and extract 234 independent pairs
gold_dir = "data_human_gold_label" if os.path.isdir("data_human_gold_label") else "04_evaluation/data_human_gold_label"
file_metrics_20 = os.path.join(gold_dir, "task_b_system_outputs_300_goldlabel_with_metrics.csv")
df = pd.read_csv(file_metrics_20, keep_default_na=False)
df_234 = df.drop_duplicates(subset=['pair_id']).copy()

# 2. Binary ground truth: yes = 1, (partial, no) = 0
df_234['y_true'] = (df_234['gold_content'] == 'yes').astype(int)

# 3. Compute length ratio
df_234['length_ratio'] = df_234['output_text'].str.split().str.len() / df_234['input_text'].str.split().str.len()

# 4. Compute ROC-AUC
auc_sim = roc_auc_score(df_234['y_true'], df_234['Sim'])
auc_len = roc_auc_score(df_234['y_true'], df_234['length_ratio'])
auc_joint = roc_auc_score(df_234['y_true'], df_234['Joint'])

# 5. Print results
print(f"Sim: {auc_sim:.4f}")
print(f"Length_Ratio: {auc_len:.4f}")
print(f"Joint: {auc_joint:.4f}")


Sim: 0.8267
Length_Ratio: 0.9073
Joint: 0.6620
